<a href="https://colab.research.google.com/github/gauravd12345/ml_proj/blob/main/rnn/rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [172]:
import re
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

# import nltk
# nltk.download('punkt_tab')
# from nltk.tokenize import word_tokenize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


In [173]:
""" Hyperparameters """

train_size = 0.8
lr = 0.1
epochs = 20
batch_size = 256
hidden_size = 200
embedding_dim = 300

In [174]:
with open("/content/tiny_shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

# text = word_tokenize(text)

# vocab = sorted(set(words))
vocab = "".join(sorted(set(text)))
vocab_len = len(vocab)

ctoi, itoc = {}, {}
for i in range(len(vocab)):
    ctoi[vocab[i]] = i
    itoc[i] = vocab[i]

print(vocab_len)

65


In [175]:
X, y = [], []
for i, j in zip(text, text[1:]):
    X.append(ctoi[i])
    y.append(ctoi[j])

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

n = len(X)
train_end = int(train_size * n)

X_train = X[:train_end]
X_test = X[train_end:]

y_train = y[:train_end]
y_test = y[train_end:]

print(X_train[:10])
print(y_train[:10])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])
tensor([47, 56, 57, 58,  1, 15, 47, 58, 47, 64])


In [176]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_len, embedding_dim)
        self.x_t = nn.Linear(embedding_dim + hidden_size, hidden_size)
        self.s_t = torch.ones(batch_size, hidden_size).to(device) * 0.1
        self.y_t = nn.Linear(hidden_size, vocab_len)


    def forward(self, w):
        w = self.embed(w)
        x = torch.cat((w, self.s_t), 1) # shape is (B, V + H)
        s = torch.sigmoid(self.x_t(x))
        self.s_t = s.detach() # detach() makes sure self.s_t isn't affected by computation graph during training
        y = self.y_t(s)
        return y

model = RNN().to(device)

In [177]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X_train, y_train)
loader = DataLoader(dataset, batch_size=batch_size, drop_last=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=lr)
for epoch in range(epochs):
    total_loss = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1, Loss: 2.5894
Epoch 2, Loss: 2.4683
Epoch 3, Loss: 2.4535
Epoch 4, Loss: 2.4472
Epoch 5, Loss: 2.4437
Epoch 6, Loss: 2.4415
Epoch 7, Loss: 2.4400
Epoch 8, Loss: 2.4389
Epoch 9, Loss: 2.4380
Epoch 10, Loss: 2.4373
Epoch 11, Loss: 2.4367
Epoch 12, Loss: 2.4362
Epoch 13, Loss: 2.4358
Epoch 14, Loss: 2.4354
Epoch 15, Loss: 2.4350
Epoch 16, Loss: 2.4347
Epoch 17, Loss: 2.4345
Epoch 18, Loss: 2.4342
Epoch 19, Loss: 2.4340
Epoch 20, Loss: 2.4338


In [188]:
import random
import torch

uppercase_chars = [c for c in "ABCDEFGHIJKLMNOPQRSTUVWXYZ" if c in ctoi]

start_char = random.choice(uppercase_chars)
data = torch.tensor([ctoi[start_char]], dtype=torch.long, device=device)

length = 100

model.eval()
model.s_t = torch.ones(1, hidden_size, device=device) * 0.1

print(start_char, end="")

with torch.no_grad():
    for i in range(1, length):
        logits = model(data)

        probs = torch.softmax(logits, dim=1)

        pred = torch.multinomial(probs, num_samples=1).item()
        next_char = itoc[pred]

        print(next_char, end="")
        if i % 80 == 0:
            print()

        data = torch.tensor([pred], dtype=torch.long, device=device)

M:

Gen he s e tave mam,
I hie s youpeio y por r.
Wecke
Abulaliro gh bou me lakil
lin-g at k's PAn he